In [2]:
import json
import pandas as pd

INPUT_PATH = r"data\kurasi_outputs\Subsidi_Tepat_BBM_Registrasi_AI_Anomali_Data_Kendaraan_Table_1_3065ee6c\20260520_123253_256939\export_csv_labeled_json_labelling.json"
OUTPUT_PATH = r"carphoto.plate_number_edit_eval.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

records = []

# Flatten nested JSON
for group_name, group_value in data.items():
    for field_name, items in group_value.items():
        for item in items:
            row = item.copy()
            row["_group"] = group_name
            row["_field"] = field_name
            records.append(row)

df = pd.DataFrame(records)

print("Jumlah label sebelum sampling:")
print(df["expected"].value_counts())

# Ambil 50 APPROVED dan 50 REJECTED
approved = df[df["expected"] == "APPROVED"].sample(n=50, random_state=42)
rejected = df[df["expected"] == "REJECTED"].sample(n=50, random_state=42)

sampled_df = pd.concat([approved, rejected], ignore_index=True)

# Acak urutan final
sampled_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Jumlah label setelah sampling:")
print(sampled_df["expected"].value_counts())

# Rebuild ke format nested awal
output_data = {}

for _, row in sampled_df.iterrows():
    group_name = row["_group"]
    field_name = row["_field"]

    if group_name not in output_data:
        output_data[group_name] = {}

    if field_name not in output_data[group_name]:
        output_data[group_name][field_name] = []

    item = row.drop(labels=["_group", "_field"]).to_dict()

    output_data[group_name][field_name].append(item)

# Save JSON
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

Jumlah label sebelum sampling:
expected
APPROVED    1216
REJECTED      53
Name: count, dtype: int64
Jumlah label setelah sampling:
expected
REJECTED    50
APPROVED    50
Name: count, dtype: int64
Saved to carphoto.plate_number_edit_eval.json


In [3]:
import pandas as pd

df = pd.read_csv(r"data\kurasi_outputs\data_train_new1800_0cec376a\20260521_084459_473492\export_csv_labeled_data_train_reviewed.csv")

df[df["ground_truth"] != df["reviewer_label"]].to_csv("data_train_diff_220526.csv")

In [6]:
import pandas as pd

df = pd.read_csv(r"data\kendaraan_230526.csv")
df_length = df.shape[0]/2

df1 = df.loc[:df_length]
df2 = df.loc[df_length:]

df1.to_csv(r"data\kendaraan_230526_part1.csv")
df2.to_csv(r"data\kendaraan_230526_part2.csv")